In [39]:
from ultralytics import YOLO
import cv2
import numpy as np

In [40]:
model = YOLO('yolo26n.pt')

In [41]:
img_path = '../data/raw/8-2-second.jpg'
img = cv2.imread(img_path)

In [42]:
result = model(img, imgsz=960 , conf=0.10)[0]


0: 736x960 28 cars, 2 motorcycles, 1 truck, 75.4ms
Speed: 6.3ms preprocess, 75.4ms inference, 0.2ms postprocess per image at shape (1, 3, 736, 960)


In [43]:
#car, motorcycle, bus, truck
VALID_CLASSES = {2, 3, 5, 7}

vehicles = []
for box in result.boxes:
    cls = int(box.cls[0])
    if cls in VALID_CLASSES:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        vehicles.append((x1, y1, x2, y2, cls))


In [44]:
vehicles.sort(key= lambda b: b[0])

widths = [x2 - x1 for (x1, y1, x2, y2, cls) in vehicles]
avg_width = np.mean(widths)

In [45]:
THRESHOLD = 1.2 * avg_width

In [46]:
for (x1, y1, x2, y2, cls) in vehicles:
    center_x = (x1 + x2) // 2
    center_y = (y1 + y2) // 2
    width = int((x2 - x1) * 0.9)
    height = int((y2 - y1) * 0.9)

    cv2.ellipse(img, (center_x, center_y), (width//2, height//2),
                0, 0, 360, (0, 255, 255), 2)

In [47]:
for i in range(len(vehicles) - 1):
    x1_a, y1_a, x2_a, y2_a, cls_a = vehicles[i]
    x1_b, y1_b, x2_b, y2_b, cls_b = vehicles[i + 1]

    center_y_a = (y1_a + y2_a) / 2
    center_y_b = (y1_b + y2_b) / 2

    gap_x = x1_b - x2_a
    gap_y = abs(center_y_b - center_y_a)

    # Dibuja las distancias para debug
    mid_x = (x2_a + x1_b) // 2
    mid_y = int((center_y_a + center_y_b) / 2)
    cv2.putText(img, f"X:{int(gap_x)} Y:{int(gap_y)}",
                (mid_x - 40, mid_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    # Criterio
    if gap_x > THRESHOLD and gap_y < avg_width * 0.5:
        x_start = x2_a
        x_end = x1_b
        y_top = min(y1_a, y1_b)
        y_bottom = max(y2_a, y2_b)

        cv2.rectangle(img, (x_start, y_top), (x_end, y_bottom), (0, 255, 0), 2)
        cv2.putText(img, "Park Here", (x_start + 5, y_top - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

In [48]:
cv2.imwrite("../data/processed/resultado9_yolo26.jpg", img)


True